# Day 2 Assignment: Vanishing Gradients in NLP

**COMP 395 – Deep Learning**

## Objectives
By the end of this assignment, you will:
1. Implement a vanilla RNN forward pass **from scratch** in PyTorch
2. Build a complete NLP pipeline: tokenization → embedding → RNN → classifier
3. Train a vanilla RNN on **text classification** using PyTorch
4. Use **gradient hooks** to measure and visualize gradient flow
5. **Observe** the vanishing gradient problem in a real training loop
6. Fix it using an **LSTM** and **gradient clipping**


---
## ⚙️ Setup

You should already have `torch` and `matplotlib` from previous assignments. No additional packages are needed — we download the dataset directly using Python's standard library.

**Time estimate:** ~25 minutes (heavily scaffolded — you fill in the key parts)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")


---
## Part 0: From-Scratch RNN Forward Pass (Warmup)

Before using `nn.RNN`, let's implement the forward pass ourselves using raw PyTorch tensors — just like we implemented gradient descent and logistic regression from scratch earlier in the course. No `nn.Module`, no autograd — just tensor operations.

Recall the RNN equations from the lecture:

$$\mathbf{h}_t = \tanh(\mathbf{W}_{hh}\,\mathbf{h}_{t-1} + \mathbf{W}_{xh}\,\mathbf{x}_t + \mathbf{b}_h)$$
$$\mathbf{y}_t = \mathbf{W}_{hy}\,\mathbf{h}_t + \mathbf{b}_y$$


In [ ]:
class RNNFromScratch:
    """
    A vanilla RNN implemented with raw PyTorch tensors (no nn.Module).
    Compare this to your from-scratch logistic regression in Assignment 4 --
    same idea, but now the hidden state feeds back into itself.
    """
    def __init__(self, input_size, hidden_size, output_size):
        # Xavier initialization (same idea as in your MLP experiments)
        scale_xh = (2.0 / (input_size + hidden_size)) ** 0.5
        scale_hh = (2.0 / (hidden_size + hidden_size)) ** 0.5
        scale_hy = (2.0 / (hidden_size + output_size)) ** 0.5

        self.W_xh = torch.randn(hidden_size, input_size) * scale_xh
        self.W_hh = torch.randn(hidden_size, hidden_size) * scale_hh
        self.b_h  = torch.zeros(hidden_size)
        self.W_hy = torch.randn(output_size, hidden_size) * scale_hy
        self.b_y  = torch.zeros(output_size)
        self.hidden_size = hidden_size

    def forward(self, inputs):
        """
        Process an entire sequence.

        Args:
            inputs: list of input tensors, each shape (input_size,)

        Returns:
            outputs: list of output tensors at each time step
            hiddens: list of hidden states (useful for inspection)
        """
        h = torch.zeros(self.hidden_size)  # h_0
        outputs = []
        hiddens = [h.clone()]

        for x_t in inputs:
            # ============================================================
            # TODO: Implement the two RNN equations from the lecture.
            # Use the weight matrices and biases defined in __init__.
            # ============================================================

            h = ...  # YOUR CODE HERE
            y = ...  # YOUR CODE HERE

            outputs.append(y)
            hiddens.append(h.clone())

        return outputs, hiddens


In [ ]:
# Test it: process a short sequence and visualize the hidden state
rnn_scratch = RNNFromScratch(input_size=8, hidden_size=32, output_size=4)

# Create a random sequence of 40 time steps
sequence = [torch.randn(8) for _ in range(40)]
outputs, hiddens = rnn_scratch.forward(sequence)

# Plot how the hidden state changes over time
norms = [h.norm().item() for h in hiddens]
plt.figure(figsize=(10, 4))
plt.plot(norms, 'b-o', markersize=3)
plt.xlabel('Time Step')
plt.ylabel('||h_t|| (Hidden State Norm)')
plt.title('From-Scratch RNN: Hidden State Norm Over Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Input shape per step:  (8,)")
print(f"Hidden state shape:    ({rnn_scratch.hidden_size},)")
print(f"Output shape per step: ({outputs[0].shape[0]},)")
print(f"Number of time steps:  {len(sequence)}")
print(f"\nNote: W_hh shape is {tuple(rnn_scratch.W_hh.shape)}")
print(f"  This is the matrix that gets 'multiplied repeatedly' during BPTT.")
print(f"  Its eigenvalues determine whether gradients vanish or explode.")
eigs = torch.linalg.eigvals(rnn_scratch.W_hh).abs()
print(f"  Max |eigenvalue| = {eigs.max():.3f}, Min |eigenvalue| = {eigs.min():.3f}")


### ✏️ Quick Check

1. How many learnable parameters does this RNN have? Count them: $\mathbf{W}_{xh}$, $\mathbf{W}_{hh}$, $\mathbf{b}_h$, $\mathbf{W}_{hy}$, $\mathbf{b}_y$.
2. Compare the forward pass to your Assignment 4 logistic regression. What's the key structural difference? (Hint: which variable persists across time steps?)
3. What happens to `h` if we feed in a sequence of 1000 steps? Does `tanh` prevent it from growing unboundedly?

Now that you understand what's happening inside, let's switch to PyTorch's `nn.RNN` and train on real data.

---


## Part 1: From Text to Tensors

Before an RNN can process text, we need to convert words into numbers. The pipeline:

1. **Tokenize**: Split text into words
2. **Build vocabulary**: Assign each unique word an integer index
3. **Numericalize**: Convert each sentence to a sequence of indices
4. **Embed**: Look up each index in a learned embedding matrix (`nn.Embedding`)

This is analogous to how we normalized pixel values for MNIST — transforming raw data into a format the network can learn from.


In [ ]:
# ========================================================
# DATASET: AG News (news article classification)
# 4 classes: World, Sports, Business, Sci/Tech
# We use this because articles are long enough to show
# vanishing gradients, and the task is straightforward.
#
# Downloaded directly as CSV — no extra packages needed.
# ========================================================

import csv
import io
import urllib.request
import os

CLASSES = ['World', 'Sports', 'Business', 'Sci/Tech']
NUM_CLASSES = 4

def download_ag_news(split='train', cache_dir='./data'):
    """Download AG News CSV and return list of (label, text) tuples."""
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, f'ag_news_{split}.csv')

    if not os.path.exists(cache_path):
        urls = [
            f'https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/{split}.csv',
            f'https://huggingface.co/datasets/fancyzhx/ag_news/resolve/main/data/{split}.csv',
        ]
        downloaded = False
        for url in urls:
            try:
                print(f"Downloading AG News {split} set...")
                urllib.request.urlretrieve(url, cache_path)
                print("Done.")
                downloaded = True
                break
            except Exception as e:
                print(f"  URL failed ({e}), trying next...")
        if not downloaded:
            raise RuntimeError(
                f"Could not download AG News {split} set. "
                f"Please download manually from:\n"
                f"  {urls[0]}\n"
                f"and save it as: {cache_path}"
            )
    else:
        print(f"Using cached AG News {split} set.")

    data = []
    with open(cache_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) >= 3:
                # CSV format: label (1-indexed), title, description
                label = int(row[0]) - 1  # shift to 0-indexed
                text = row[1] + ' ' + row[2]  # combine title + description
                data.append((label, text))
    return data

train_raw = download_ag_news('train')
test_raw = download_ag_news('test')

# Subsample for speed (full dataset is 120k train, 7.6k test)
MAX_TRAIN = 4000
MAX_TEST = 800

np.random.shuffle(train_raw)
np.random.shuffle(test_raw)
train_raw = train_raw[:MAX_TRAIN]
test_raw = test_raw[:MAX_TEST]

print(f"\nTrain samples: {len(train_raw)}, Test samples: {len(test_raw)}")
print(f"Classes: {CLASSES}")
print(f"\nExample:")
print(f"  Label: {train_raw[0][0]} ({CLASSES[train_raw[0][0]]})")
print(f"  Text:  '{train_raw[0][1][:100]}...'")


### Building a Vocabulary

We need to map words to integer indices. Two special tokens:
- `<pad>` (index 0): pads short sequences to equal length in a batch
- `<unk>` (index 1): stands in for words not seen during training


In [ ]:
# Simple whitespace + lowercase tokenizer
def tokenize(text):
    """Basic tokenizer: lowercase and split on whitespace."""
    return text.lower().split()

# Build vocabulary from training data
def build_vocab(data, max_vocab=10000):
    """
    Count word frequencies and keep the top max_vocab words.
    Returns: word_to_idx dict
    """
    counter = Counter()
    for label, text in data:
        counter.update(tokenize(text))

    # Special tokens
    vocab = {'<pad>': 0, '<unk>': 1}
    for word, count in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)

    return vocab

def tokenize_and_numericalize(text, vocab, max_length=200):
    """Convert text string to list of integer indices."""
    tokens = tokenize(text)[:max_length]
    indices = [vocab.get(t, vocab['<unk>']) for t in tokens]
    return indices

vocab = build_vocab(train_raw)
VOCAB_SIZE = len(vocab)

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"\nExample tokenization:")
sample_label, sample_text = train_raw[0]
sample_indices = tokenize_and_numericalize(sample_text, vocab)
print(f"  Text (first 60 chars): '{sample_text[:60]}...'")
print(f"  Indices (first 10):    {sample_indices[:10]}")
print(f"  Sequence length:       {len(sample_indices)}")


In [ ]:
# Create PyTorch Dataset
class TextClassificationDataset(Dataset):
    def __init__(self, data, vocab, max_length=200):
        self.samples = []
        for label, text in data:
            indices = tokenize_and_numericalize(text, vocab, max_length)
            if len(indices) > 0:
                self.samples.append((
                    torch.tensor(indices, dtype=torch.long),
                    label
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    """Pad sequences to the same length within a batch."""
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.tensor(labels, dtype=torch.long)
    return texts_padded.to(device), labels.to(device)

train_dataset = TextClassificationDataset(train_raw, vocab)
test_dataset = TextClassificationDataset(test_raw, vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

# Show sequence length distribution
seq_lengths = [len(s[0]) for s in train_dataset]
print(f"Sequence length stats:")
print(f"  Min: {min(seq_lengths)}, Max: {max(seq_lengths)}, "
      f"Mean: {np.mean(seq_lengths):.0f}, Median: {np.median(seq_lengths):.0f}")
print(f"\nTraining batches: {len(train_loader)}, Test batches: {len(test_loader)}")


---
## Part 2: Building the Text Classifier

Our model architecture:

```
Input token indices → Embedding → RNN/LSTM → Last hidden state → Linear → Class prediction
```

The **Embedding layer** (`nn.Embedding`) is a lookup table that maps each word index to a learned dense vector. Think of it as a learnable first layer that takes discrete inputs (integers) rather than continuous ones.

Compare this to your CNN workflow: there you used `transforms.Normalize()` to preprocess pixel values. Here, the embedding layer *learns* the preprocessing — it figures out which words are similar as part of training.


In [ ]:
class TextClassifier(nn.Module):
    """
    A text classifier using either a vanilla RNN or LSTM.

    Architecture:
        Token indices -> Embedding -> RNN/LSTM -> Last hidden state -> Linear -> Output

    Compare to your CNN: Conv blocks extracted spatial features from images.
    Here, the RNN extracts sequential features from text.
    """
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes,
                 num_layers=2, cell_type='RNN', pad_idx=0):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.cell_type = cell_type

        # Embedding layer: word index -> dense vector
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # Recurrent layer
        if cell_type == 'RNN':
            self.rnn = nn.RNN(embed_dim, hidden_size, num_layers,
                              batch_first=True, nonlinearity='tanh')
        elif cell_type == 'LSTM':
            self.rnn = nn.LSTM(embed_dim, hidden_size, num_layers,
                               batch_first=True)
        elif cell_type == 'GRU':
            self.rnn = nn.GRU(embed_dim, hidden_size, num_layers,
                              batch_first=True)

        # Output classifier (like the FC head after your CNN conv blocks)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x shape: (batch, seq_len) — integer token indices
        embedded = self.embedding(x)        # (batch, seq_len, embed_dim)
        out, hidden = self.rnn(embedded)    # out: (batch, seq_len, hidden_size)

        # Take the LAST time step's output as the sequence representation
        last_output = out[:, -1, :]         # (batch, hidden_size)
        logits = self.classifier(last_output)  # (batch, num_classes)
        return logits


### ✏️ Think-Pair-Share: Shape Reasoning

You've traced tensor shapes through CNNs (N,C,H,W). Now trace them through this model:

1. If `batch_size=64`, `seq_len=150`, and `embed_dim=64`, what's the shape after `self.embedding(x)`?
2. After `self.rnn(embedded)`, `out` has shape `(64, 150, hidden_size)`. Why does the sequence dimension persist?
3. We take `out[:, -1, :]` — the **last** time step. What information from earlier time steps is "compressed" into this single vector? What might be lost?


---
## Part 3: Gradient Hooks — Seeing Inside the Black Box

In your MLP and CNN assignments, you used `loss.backward()` and trusted autograd to compute gradients. PyTorch lets us **register hooks** on any parameter to actually *capture* those gradients during backpropagation.

A **backward hook** fires every time gradients flow through a parameter. We'll record the gradient norm at each RNN layer to see if early layers receive smaller gradients — the hallmark of the vanishing gradient problem you first encountered with sigmoid in hidden layers.


In [ ]:
class GradientTracker:
    """
    Registers backward hooks on model parameters to track gradient norms
    during training. This lets us visualize gradient flow.
    """
    def __init__(self, model):
        self.gradient_norms = {}  # {param_name: [norm_per_step]}
        self.hooks = []

        for name, param in model.named_parameters():
            if param.requires_grad:
                self.gradient_norms[name] = []
                # Register a hook that fires during the backward pass
                hook = param.register_hook(
                    self._make_hook(name)
                )
                self.hooks.append(hook)

    def _make_hook(self, name):
        def hook(grad):
            self.gradient_norms[name].append(grad.norm().item())
        return hook

    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()

    def get_layer_gradient_summary(self):
        """Return mean gradient norm per parameter (last 10 steps)."""
        summary = {}
        for name, norms in self.gradient_norms.items():
            if len(norms) > 0:
                summary[name] = np.mean(norms[-10:])
        return summary


---
## Part 4: Training and Observing Gradients

The training loop below follows the same pattern you've used since Assignment 2:
`zero_grad → forward → loss → backward → step`. The only new piece is the `GradientTracker` recording norms at each step.


In [ ]:
def train_and_track(model, train_loader, test_loader, num_epochs=8,
                     lr=0.001, clip_grad=None):
    """
    Train the model and track gradient norms at every step.

    Args:
        clip_grad: If not None, clip gradients to this max norm.
    """
    criterion = nn.CrossEntropyLoss()  # same as your MNIST CNN
    optimizer = optim.Adam(model.parameters(), lr=lr)
    tracker = GradientTracker(model)

    train_losses = []
    test_accs = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            output = model(batch_x)
            loss = criterion(output, batch_y)
            loss.backward()

            # Optional: gradient clipping
            if clip_grad is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)

        # Test accuracy
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                output = model(batch_x)
                _, predicted = torch.max(output, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        acc = correct / total
        test_accs.append(acc)

        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | "
              f"Test Acc: {acc:.3f}")

    tracker.remove_hooks()
    return train_losses, test_accs, tracker.gradient_norms


### 🚀 Experiment 1: Train a Vanilla RNN


In [ ]:
# Hyperparameters
EMBED_DIM = 64
HIDDEN_SIZE = 64

# Create a vanilla RNN model
rnn_model = TextClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    num_classes=NUM_CLASSES,
    num_layers=2,
    cell_type='RNN'
).to(device)

print(f"Model: Vanilla RNN")
total_params = sum(p.numel() for p in rnn_model.parameters())
rnn_params = sum(p.numel() for n, p in rnn_model.named_parameters() if 'rnn' in n)
embed_params = sum(p.numel() for n, p in rnn_model.named_parameters() if 'embedding' in n)
print(f"Total parameters:     {total_params:,}")
print(f"  Embedding params:   {embed_params:,}")
print(f"  RNN params:         {rnn_params:,}")
print()

rnn_losses, rnn_accs, rnn_grad_norms = train_and_track(
    rnn_model, train_loader, test_loader, num_epochs=8
)


### 📊 Visualizing Gradient Flow

Now let's look at the gradient norms across different layers. If vanishing gradients are present, we expect **earlier RNN layers to have much smaller gradients** than later layers.

We focus on the `weight_hh` (hidden-to-hidden) matrices — these are the matrices multiplied repeatedly during BPTT, and are where vanishing/exploding gradients manifest.


In [ ]:
def plot_gradient_comparison(grad_norms, title="Gradient Norms by Layer"):
    """
    Plot gradient norms for RNN weight matrices.
    Focus on weight_hh (hidden-to-hidden) matrices at each layer.
    """
    hh_layers = {name: norms for name, norms in grad_norms.items()
                 if 'weight_hh' in name}

    if not hh_layers:
        print("No weight_hh layers found. Showing all RNN layers.")
        hh_layers = {n: v for n, v in grad_norms.items() if 'rnn' in n}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Gradient norms over training steps
    for name, norms in sorted(hh_layers.items()):
        window = min(10, len(norms) // 5) if len(norms) > 20 else 1
        if window > 1:
            smoothed = np.convolve(norms, np.ones(window)/window, mode='valid')
        else:
            smoothed = norms
        axes[0].plot(smoothed, label=name.replace('rnn.', ''), alpha=0.8)
    axes[0].set_xlabel('Training Step')
    axes[0].set_ylabel('Gradient Norm')
    axes[0].set_title(f'{title}\n(over training steps)')
    axes[0].legend(fontsize=8)
    axes[0].set_yscale('log')
    axes[0].grid(True, alpha=0.3)

    # Plot 2: Average gradient norm per layer (bar chart)
    names = sorted(hh_layers.keys())
    avg_norms = [np.mean(hh_layers[n][-20:]) for n in names]
    colors = ['#ff6b6b' if 'l0' in n or '_0' in n.split('weight')[0]
              else '#4ecdc4' for n in names]
    axes[1].bar(range(len(names)), avg_norms, color=colors)
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels([n.replace('rnn.', '') for n in names],
                            rotation=45, ha='right', fontsize=8)
    axes[1].set_ylabel('Average Gradient Norm')
    axes[1].set_title(f'{title}\n(average of last 20 steps)')
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

    # Print ratio analysis
    if len(avg_norms) >= 2:
        ratio = avg_norms[0] / avg_norms[-1] if avg_norms[-1] > 0 else float('inf')
        print(f"\nGradient ratio (first layer / last layer): {ratio:.4f}")
        if ratio < 0.1:
            print(f"⚠️  Vanishing gradients detected! First layer gets ~{1/ratio:.1f}x LESS gradient.")
        elif ratio > 10:
            print("⚠️  Exploding gradients detected!")
        else:
            print("✅ Gradients are relatively stable across layers.")

plot_gradient_comparison(rnn_grad_norms, "Vanilla RNN — Gradient Norms")


### ✏️ Questions — Pause and Reflect

1. **Look at the bar chart.** Is the first layer's gradient much smaller than the last layer's? By what factor?
2. **Why does this happen?** Connect what you see to the matrix power analysis from Day 1 — the gradient must flow backward through the `weight_hh` matrix at every time step.
3. **Compare to sigmoid vanishing gradients in MLPs.** In Lab 4, sigmoid squashed gradients because $\sigma'(x) \leq 0.25$. Here, `tanh` has $\tanh'(x) \leq 1.0$. So why do gradients *still* vanish?

---


### 🚀 Experiment 2: Train an LSTM

Now let's swap the vanilla RNN for an LSTM. Same data, same hyperparameters — only the recurrent cell type changes.


In [ ]:
lstm_model = TextClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    num_classes=NUM_CLASSES,
    num_layers=2,
    cell_type='LSTM'
).to(device)

print(f"Model: LSTM")
lstm_total = sum(p.numel() for p in lstm_model.parameters())
lstm_rnn = sum(p.numel() for n, p in lstm_model.named_parameters() if 'rnn' in n)
print(f"Total parameters: {lstm_total:,}  (RNN part: {lstm_rnn:,})")
print(f"Note: LSTM has ~4x more RNN parameters than vanilla RNN.")
print(f"  (Why? It has 4 gate matrices instead of 1 weight matrix.)\n")

lstm_losses, lstm_accs, lstm_grad_norms = train_and_track(
    lstm_model, train_loader, test_loader, num_epochs=8
)


In [ ]:
plot_gradient_comparison(lstm_grad_norms, "LSTM — Gradient Norms")


### 📊 Head-to-Head Comparison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss comparison
axes[0].plot(rnn_losses, 'r-o', label='Vanilla RNN', markersize=5)
axes[0].plot(lstm_losses, 'b-s', label='LSTM', markersize=5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss: RNN vs LSTM')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
axes[1].plot(rnn_accs, 'r-o', label='Vanilla RNN', markersize=5)
axes[1].plot(lstm_accs, 'b-s', label='LSTM', markersize=5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Test Accuracy: RNN vs LSTM')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f"\nFinal test accuracy — RNN: {rnn_accs[-1]:.3f}, LSTM: {lstm_accs[-1]:.3f}")


---
### 🚀 Experiment 3: Gradient Clipping on the Vanilla RNN

Gradient clipping caps the gradient norm during training. It's the standard fix for **exploding** gradients. But does it help with **vanishing** gradients?


In [ ]:
rnn_clipped = TextClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    num_classes=NUM_CLASSES,
    num_layers=2,
    cell_type='RNN'
).to(device)

print("Training Vanilla RNN with gradient clipping (max_norm=1.0)\n")

clip_losses, clip_accs, clip_grad_norms = train_and_track(
    rnn_clipped, train_loader, test_loader, num_epochs=8, clip_grad=1.0
)


In [ ]:
# Three-way comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rnn_losses, 'r-o', label='RNN (no clip)', markersize=4)
axes[0].plot(clip_losses, 'm-^', label='RNN (clipped)', markersize=4)
axes[0].plot(lstm_losses, 'b-s', label='LSTM', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(rnn_accs, 'r-o', label='RNN (no clip)', markersize=4)
axes[1].plot(clip_accs, 'm-^', label='RNN (clipped)', markersize=4)
axes[1].plot(lstm_accs, 'b-s', label='LSTM', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Test Accuracy Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f"\nFinal accuracies:")
print(f"  Vanilla RNN:          {rnn_accs[-1]:.3f}")
print(f"  RNN + Grad Clipping:  {clip_accs[-1]:.3f}")
print(f"  LSTM:                 {lstm_accs[-1]:.3f}")


---
## Part 5: Reflection Questions

Answer these in the Markdown cells below (double-click to edit).

### Q1: Vanishing Gradients Observed
Describe what you observed in the gradient norm plots for the vanilla RNN. How did the gradient magnitude at the first layer compare to the last layer? Why does this happen? (Use the chain rule and the matrix power intuition from Day 1.)


*Your answer here*


### Q2: LSTM Improvement
How did the LSTM's gradient flow differ from the vanilla RNN? Why does the LSTM's architecture (cell state + gates) help? Connect your answer to the "conveyor belt" metaphor from the lecture — specifically, why is an **additive** gradient path better than a **multiplicative** one?


*Your answer here*


### Q3: Gradient Clipping — Necessary but Not Sufficient
Did gradient clipping improve the vanilla RNN's accuracy? Why is clipping effective against *exploding* gradients but not *vanishing* gradients? (Hint: What does clipping do to a gradient that's already near zero?)


*Your answer here*


### Q4: Comparing Gradient Problems Across Architectures
You've now seen vanishing gradients in two contexts:
1. **Sigmoid in MLP hidden layers** (Lab 4): $\sigma'(x) \leq 0.25$, so gradients shrink at every layer.
2. **Vanilla RNNs over long sequences**: gradients shrink across time steps.

Are these the same problem or different problems? Is the LSTM solution analogous to the ReLU solution for MLPs, or is it fundamentally different? Explain your reasoning.


*Your answer here*


---
## Extensions (Optional — if you finish early)

### Extension 1: GRU Comparison
Add a GRU model (`cell_type='GRU'`) to the experiment. How does it compare to the RNN and LSTM in terms of gradient flow and accuracy? Log all three to MLflow for a clean comparison.

### Extension 2: Sequence Length Experiment
Truncate texts to `max_length=50` vs `max_length=300`. Does the vanilla RNN degrade more on longer sequences? Does the LSTM hold up better? This directly tests the long-range dependency prediction from the lecture.

### Extension 3: Deeper Networks
Try `num_layers=4` instead of 2. Does the vanishing gradient problem get worse with more layers? Plot and compare. How does this relate to the depth experiments you ran on MLPs?

### Extension 4: From-Scratch Backward Pass (Challenge)
Extend the `RNNFromScratch` class from Part 0 to include a backward pass. Compute $\frac{\partial \mathcal{L}}{\partial \mathbf{W}_{hh}}$ by hand using BPTT, then implement it. Compare your hand-computed gradients to PyTorch autograd on the same inputs to verify correctness.
